# 06. 주식 옵션: 현물·배당·IV 서페이스, 엔진 비교, Heston, SSR

| 단계 | 코드 |
|---|---|
| 주식 설정 | `configs/equity/spx.yaml` |
| 시장 객체 | `equity.EquitySpot`, `VolSurface`, 배당 = `curves.CurveFactors` |
| 파생 객체 | `equity.HestonModelObject` (`depends_on`, `recalibrate_on`) |
| 상품·엔진 | `portfolio.EquityOption`, `ENGINES` (AnalyticBS, FdBS, AnalyticHeston) |
| 연동 규칙·민감도 | `scenarios.ssr`, `risk.factor_delta` |

> 데이터: `tests/fixtures/market_sample.txt`는 **합성 샘플**이다. 흐름을 익히는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

from quanttool.config import load_config
from quanttool.market import build_market
from quanttool.market.quotes import read_quotes
from quanttool.portfolio import Portfolio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
cfg = load_config(ROOT / "configs")
quotes = read_quotes(ROOT / "tests" / "fixtures" / "market_sample.txt")  # 합성 샘플
ASOF = date(2026, 10, 23)
market = build_market(ASOF, quotes[ASOF], cfg)

spot, vol = market.objects["SPX"], market.objects["SPX_VOL"]
print("현물", spot.value(), "| 요인 수", sum(k.startswith("SPX") for k in market.factors()))
pd.DataFrame(
    vol.base_vols,
    index=pd.Index(vol.strike_pcts, name="strike %"),
    columns=pd.Index(vol.expiries, name="expiry"),
)

## 1. 시장 객체 셋과 파생 객체 하나
- `SPX`: 현물. 요인 `SPX/spot`은 **상대** 이동 (0.01 = +1%).
- `SPX_DIV`: 평탄 배당률 커브. 금리 커브와 같은 `CurveFactors`라 요인도 `SPX_DIV/1Y`처럼 같다.
- `SPX_VOL`: 격자 IV → `BlackVarianceSurface`. 요인 `SPX_VOL/1Y/90`은 격자 vol 이동. 행사가는 asof 현물 기준 **절대값**.
- `SPX_HESTON`: 위 넷(할인 UST 포함)에 기대는 **파생 객체**. 시장이 의존 순서로 refresh한다.

In [ ]:
market._order, market.objects["SPX_HESTON"].depends_on

## 2. 같은 옵션, 다른 엔진
상품 조립(`build_equity_option`)은 하나, 엔진은 `pricing.yaml` 기본값(`AnalyticBS`)이나 거래의 `engine` 필드로 고른다. Heston은 서페이스에 캘리브레이션했으므로 BS 서페이스 가격과 가깝다(합성 IV는 Heston 가격에서 만들었다).

In [ ]:
trades = list(Portfolio.from_file(ROOT / "positions" / "spx_options.yaml").trades.values())


def book_with(engine, **kw):
    rows = [{**t.model_dump(), "engine": engine, **kw} for t in trades]
    return Portfolio.from_records(rows).build(market, cfg)


books = {
    "AnalyticBS": book_with("AnalyticBS"),
    "FdBS": book_with("FdBS"),
    "AnalyticHeston": book_with("AnalyticHeston", model="SPX_HESTON"),
}
pd.DataFrame({n: b.npvs() for n, b in books.items()})

## 3. Heston 캘리브레이션: 시세를 만든 파라미터를 되찾는다

In [ ]:
h = market.objects["SPX_HESTON"]
true = {"theta": 0.045, "kappa": 1.8, "sigma": 0.6, "rho": -0.7, "v0": 0.033}
print("최대 헬퍼 가격 오차", max(map(abs, h.calibration_errors())))
pd.DataFrame({"calibrated": h.params(), "generating": true})

## 4. 재캘리브레이션 정책 (`recalibrate_on`)
`spx.yaml`은 `recalibrate_on: [SPX, SPX_VOL]`: 현물·vol이 바뀌면 다시 캘리브레이션, 금리·배당은 파라미터 고정 (프로세스 Handle로 커브는 따라온다). 같은 입력은 캐시한다. 비우면 언제나 고정.

In [ ]:
from quanttool.analysis.scenarios import parallel

vol_keys = [k for k in market.factors() if k.startswith("SPX_VOL/")]
rows = {"base": h.params()}
for name, s in {
    "vol +1%": dict.fromkeys(vol_keys, 0.01),
    "spot +5%": {"SPX/spot": 0.05},
    "UST +10bp": parallel(market, "UST", 1e-3),
}.items():
    with market.scenario(s):
        rows[name] = h.params()
pd.DataFrame(rows).T

고정 정책이면 vol을 흔들어도 Heston 가격은 현물·커브로만 움직인다. 두 정책의 vol +1% 손익:

In [ ]:
spx = cfg.equities["SPX"]
fixed_cfg = cfg.model_copy(
    update={
        "equities": {
            "SPX": spx.model_copy(
                update={"heston": spx.heston.model_copy(update={"recalibrate_on": []})}
            )
        }
    }
)
fixed_market = build_market(ASOF, quotes[ASOF], fixed_cfg)
fixed_book = Portfolio.from_records(
    [{**t.model_dump(), "engine": "AnalyticHeston", "model": "SPX_HESTON"} for t in trades]
).build(fixed_market, fixed_cfg)
from quanttool.analysis.risk import scenario_pnl

pd.DataFrame(
    {
        "recalibrate": scenario_pnl(
            books["AnalyticHeston"], market, {"vol +1%": dict.fromkeys(vol_keys, 0.01)}
        ).iloc[0],
        "fixed": scenario_pnl(
            fixed_book, fixed_market, {"vol +1%": dict.fromkeys(vol_keys, 0.01)}
        ).iloc[0],
        "BS": scenario_pnl(
            books["AnalyticBS"], market, {"vol +1%": dict.fromkeys(vol_keys, 0.01)}
        ).iloc[0],
    }
)

## 5. SSR: 현물과 vol을 함께 움직이는 규칙
`ssr(market, 'SPX', 'SPX_VOL', r, x)`는 `{SPX/spot: r, SPX_VOL/T/K: Δσ}` 사전이다.

Δσ(T, K) = σ(T, K·(1+r)^(x−1)) − σ(T, K). x=1 sticky strike(vol 그대로), x=0 sticky moneyness.

`factor_delta`는 요인 키나 이런 함수를 받아 중심 차분 delta·gamma를 낸다. 1% 현물 이동당 손익:

In [ ]:
from quanttool.analysis.risk import factor_delta
from quanttool.analysis.scenarios import ssr

bs = books["AnalyticBS"]
moves = {
    f"ssr={x}": (lambda r, x=x: ssr(market, "SPX", "SPX_VOL", r, x)) for x in [0.0, 0.5, 1.0, 1.5]
}
moves["vega (격자 +1vol)"] = lambda r: dict.fromkeys(vol_keys, r)
out = factor_delta(bs, market, moves, 0.01)
out["delta"].round(0)

읽는 법: skew가 음수(낮은 행사가 vol이 높다)라 sticky moneyness(x=0)에서는 현물 상승 시 고정 행사가 vol이 오른다. 그래서 vega가 양수인 롱 콜의 delta가 sticky strike보다 크다. QuantLib `delta()`는 x=1(vol 고정)과 같다.

In [ ]:
with market.active():
    ql_delta = {
        i: bs.instruments[i].delta() * t.quantity * spot.value() * 0.01
        for i, t in bs.trades.items()
    }
pd.DataFrame({"ssr=1": out["delta"].loc["ssr=1.0"], "QuantLib delta × 1%": ql_delta})

## 6. 시간: theta
vol 서페이스는 커브와 같은 ForwardForward로 굴린다: 오늘 서페이스의 선도 분산이 실현된다. Heston은 굴린 시장(서페이스 포함)에 다시 캘리브레이션한다.

In [ ]:
from quanttool.analysis.risk import theta

pd.DataFrame({n: theta(b, market, date(2026, 10, 26)) for n, b in books.items()})